# 01 — Data Exploration

**Team:** StellarX  
**Phase:** 1 — Dataset and Star-Field Generation  
**Status:** ✅ Fully executable — Phase 1 implementation complete.

## What this notebook covers

1. Load and inspect the Hipparcos bright-star catalog
2. Visualise the catalog: sky coverage, magnitude distribution
3. Generate sample synthetic star-field images
4. Display generated images with ground-truth star overlays
5. Inspect brightness distributions across the dataset
6. Examine the metadata schema
7. Run a small full-pipeline generation and verify output

## Prerequisites

```bash
# From the repository root:
pip install -r requirements.txt
```

All cells assume the notebook is run from the **repository root**, not from `notebooks/`.
If running from `notebooks/`, uncomment the `sys.path` cell below.

In [ ]:
# Uncomment if running from inside the notebooks/ directory
# import sys, pathlib
# sys.path.insert(0, str(pathlib.Path.cwd().parent))

import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import yaml

# Suppress minor warnings during exploration
warnings.filterwarnings('ignore')

# Non-interactive backend — safe for both Jupyter and headless runs
matplotlib.use('Agg')
%matplotlib inline

print(f'Python  : {sys.version.split()[0]}')
print(f'NumPy   : {np.__version__}')
print(f'Pandas  : {pd.__version__}')
print(f'Matplotlib: {matplotlib.__version__}')

In [ ]:
# Load project configuration
with open('config.yaml', 'r') as f:
    config = yaml.safe_load(f)

ds_cfg = config['dataset']
print('Configuration loaded.')
print(f"  Catalog file : {ds_cfg['catalog_file']}")
print(f"  Image size   : {ds_cfg['image_width']} x {ds_cfg['image_height']} px")
print(f"  FoV          : {ds_cfg['field_of_view_deg']}°")
print(f"  Random seed  : {ds_cfg['random_seed']}")
print(f"  Splits       : train={ds_cfg['num_train']}  val={ds_cfg['num_val']}  test={ds_cfg['num_test']}")

---
## 1. Load and inspect the star catalog

In [ ]:
from src.catalog.catalog_loader import load_catalog

catalog = load_catalog(ds_cfg['catalog_file'], config)
print(catalog)

summary = catalog.summary()
print(f"\nCatalog summary:")
for k, v in summary.items():
    print(f"  {k:15s}: {v}")

In [ ]:
# Show the catalog as a DataFrame for easy inspection
rows = []
for star in catalog:
    rows.append({
        'star_id':       star.star_id,
        'ra_deg':        round(star.ra_deg, 4),
        'dec_deg':       round(star.dec_deg, 4),
        'vmag':          star.magnitude,
        'common_name':   star.metadata.get('common_name', ''),
        'spectral_type': star.metadata.get('spectral_type', ''),
    })

df = pd.DataFrame(rows).sort_values('vmag').reset_index(drop=True)
print(f'Total entries : {len(df)}')
df.head(15)

---
## 2. Catalog visualisation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Sky coverage (RA / Dec scatter) ──────────────────────────────────
ax = axes[0]
sc = ax.scatter(
    df['ra_deg'], df['dec_deg'],
    c=df['vmag'], cmap='viridis_r',
    s=np.clip(100 * 10**(-0.4 * df['vmag']), 5, 200),
    alpha=0.85, edgecolors='none'
)
plt.colorbar(sc, ax=ax, label='V magnitude')
ax.set_xlabel('Right Ascension (deg)')
ax.set_ylabel('Declination (deg)')
ax.set_title('Sky Coverage — Hipparcos Bright Star Subset\n'
             '(SYNTHETIC DATA FOUNDATION — not real imagery)')
ax.set_xlim(0, 360)
ax.set_ylim(-90, 90)
ax.invert_xaxis()   # astronomical convention
ax.grid(True, alpha=0.3)

# ── V magnitude distribution ──────────────────────────────────────────
ax = axes[1]
ax.hist(df['vmag'], bins=20, color='steelblue', edgecolor='white', linewidth=0.5)
ax.axvline(df['vmag'].mean(), color='tomato', linestyle='--', label=f"Mean = {df['vmag'].mean():.2f}")
ax.set_xlabel('V magnitude')
ax.set_ylabel('Count')
ax.set_title('V Magnitude Distribution')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('notebooks/fig_01_catalog_overview.png', dpi=120, bbox_inches='tight')
plt.show()
print('Figure saved to notebooks/fig_01_catalog_overview.png')

---
## 3. Generate sample synthetic star-field images

> **Note:** These are **synthetic star-field images** generated from the Hipparcos
> catalog using a simplified spacecraft star-sensor simulation. They are
> **not real astronomical photographs or spacecraft imagery**.

In [ ]:
from src.preprocessing.star_field_generator import StarFieldGenerator

# Use a slightly larger image for notebook visualisation
vis_cfg = {**ds_cfg, 'image_width': 256, 'image_height': 256}
generator = StarFieldGenerator(catalog, vis_cfg)

# Generate 6 sample images with fixed seeds for reproducibility
sample_seeds = [42, 123, 999, 2024, 7777, 31415]
samples = [generator.generate(seed=s) for s in sample_seeds]

print(f'Generated {len(samples)} sample images.')
for sf in samples:
    print(f'  seed={sf.seed:6d}  '
          f'boresight=({sf.boresight_ra_deg:7.2f}°, {sf.boresight_dec_deg:+7.2f}°)  '
          f'roll={sf.roll_deg:6.1f}°  '
          f'n_stars={len(sf.stars):2d}  '
          f'img_max={sf.image.max():.4f}')

---
## 4. Display generated images with ground-truth star overlays

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for i, sf in enumerate(samples):
    ax = axes[i]

    # Display image (log-stretch for visibility)
    display_img = np.log1p(sf.image * 50) / np.log1p(50)
    ax.imshow(display_img, cmap='gray', vmin=0, vmax=1, origin='upper')

    # Overlay ground-truth star positions
    if sf.stars:
        xs = [s.x_px for s in sf.stars]
        ys = [s.y_px for s in sf.stars]
        fluxes = [s.flux for s in sf.stars]
        sizes = [max(20, f * 120) for f in fluxes]
        ax.scatter(xs, ys, s=sizes, facecolors='none',
                   edgecolors='lime', linewidths=0.8, alpha=0.9)

        # Label the brightest star
        brightest = max(sf.stars, key=lambda s: s.flux)
        ax.annotate(
            brightest.star_id,
            (brightest.x_px, brightest.y_px),
            color='yellow', fontsize=6,
            xytext=(4, 4), textcoords='offset points'
        )

    ax.set_title(
        f'seed={sf.seed}  n={len(sf.stars)} stars\n'
        f'RA={sf.boresight_ra_deg:.1f}° Dec={sf.boresight_dec_deg:+.1f}°',
        fontsize=8
    )
    ax.axis('off')

# Legend
legend_patch = mpatches.Patch(color='none', label='○ = ground-truth star position (lime circle)')
fig.legend(handles=[legend_patch], loc='lower center', fontsize=9, framealpha=0.8)

fig.suptitle(
    'Synthetic Star-Field Images — StellarX Phase 1\n'
    '[SYNTHETIC DATA — Hipparcos catalog + simulated star-sensor noise]',
    fontsize=11, y=1.01
)
plt.tight_layout()
plt.savefig('notebooks/fig_02_sample_starfields.png', dpi=120, bbox_inches='tight')
plt.show()
print('Figure saved to notebooks/fig_02_sample_starfields.png')

---
## 5. Brightness distributions

In [ ]:
# Collect per-star statistics across all sample images
all_flux   = [s.flux  for sf in samples for s in sf.stars]
all_vmag   = [s.vmag  for sf in samples for s in sf.stars]
all_n_stars = [len(sf.stars) for sf in samples]
all_img_max = [sf.image.max() for sf in samples]

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Normalised flux histogram
ax = axes[0]
ax.hist(all_flux, bins=20, color='cornflowerblue', edgecolor='white', linewidth=0.5)
ax.set_xlabel('Normalised Flux (brightest star = 1.0)')
ax.set_ylabel('Count')
ax.set_title('Per-Star Flux Distribution')
ax.grid(True, alpha=0.3)

# V magnitude histogram
ax = axes[1]
ax.hist(all_vmag, bins=20, color='mediumseagreen', edgecolor='white', linewidth=0.5)
ax.set_xlabel('V Magnitude')
ax.set_ylabel('Count')
ax.set_title('V Magnitude of Rendered Stars')
ax.grid(True, alpha=0.3)

# Stars per image
ax = axes[2]
ax.bar(range(len(samples)), all_n_stars, color='salmon', edgecolor='white')
ax.set_xlabel('Sample index')
ax.set_ylabel('Stars in frame')
ax.set_title('Number of Stars per Generated Image')
ax.set_xticks(range(len(samples)))
ax.set_xticklabels([f'seed={s}' for s in sample_seeds], rotation=30, ha='right', fontsize=7)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('notebooks/fig_03_brightness_distributions.png', dpi=120, bbox_inches='tight')
plt.show()

print(f'\nAggregate stats across {len(samples)} images:')
print(f'  Total stars rendered : {len(all_flux)}')
print(f'  Stars per image      : min={min(all_n_stars)}  max={max(all_n_stars)}  mean={np.mean(all_n_stars):.1f}')
print(f'  Flux range           : {min(all_flux):.4f} – {max(all_flux):.4f}')
print(f'  V mag range          : {min(all_vmag):.2f} – {max(all_vmag):.2f}')
print(f'  Image max pixel      : {min(all_img_max):.4f} – {max(all_img_max):.4f}')

---
## 6. Metadata schema inspection

In [ ]:
from src.preprocessing.dataset_builder import _build_metadata_entry

# Show the metadata for the first sample as a pretty-printed dict
import json
sf0 = samples[0]
entry = _build_metadata_entry(sf0, split='train', rel_path='train/000000.png', seed=sf0.seed)

print('Metadata schema for one sample:')
print(json.dumps(entry, indent=2)[:2000])   # truncate if many stars

In [ ]:
# Schema table for documentation
schema = [
    ('sample_id',         'str',   'Unique sample identifier, e.g. "train_000042"'),
    ('split',             'str',   '"train" | "val" | "test"'),
    ('image_file',        'str',   'Relative path from output_dir, e.g. "train/000042.png"'),
    ('seed',              'int',   'Random seed used to generate this image'),
    ('image_width',       'int',   'Image width in pixels'),
    ('image_height',      'int',   'Image height in pixels'),
    ('fov_deg',           'float', 'Full field-of-view in degrees'),
    ('boresight_ra_deg',  'float', 'Camera boresight RA in degrees (J2000 ICRS)'),
    ('boresight_dec_deg', 'float', 'Camera boresight Dec in degrees (J2000 ICRS)'),
    ('roll_deg',          'float', 'Camera roll angle around boresight in degrees'),
    ('n_stars',           'int',   'Number of stars rendered into this image'),
    ('stars[].star_id',   'str',   'Hipparcos identifier, "HIP_<n>"'),
    ('stars[].x_px',      'float', 'Star centroid column (horizontal pixel coordinate)'),
    ('stars[].y_px',      'float', 'Star centroid row (vertical pixel coordinate)'),
    ('stars[].flux',      'float', 'Normalised flux in (0, 1]; 1.0 = brightest in frame'),
    ('stars[].vmag',      'float', 'Johnson V-band apparent magnitude'),
    ('stars[].ra_deg',    'float', 'Catalog RA in degrees'),
    ('stars[].dec_deg',   'float', 'Catalog Dec in degrees'),
]

schema_df = pd.DataFrame(schema, columns=['Field', 'Type', 'Description'])
print(schema_df.to_string(index=False))

---
## 7. Small full-pipeline generation test

Runs `build_dataset` with a very small count (3+1+1 images) to validate
the full pipeline end-to-end: generation → PNG save → metadata JSON → reload.

In [ ]:
import tempfile
from pathlib import Path
from src.preprocessing.dataset_builder import build_dataset, load_metadata
from src.preprocessing.image_preprocessing import load_image

with tempfile.TemporaryDirectory(prefix='stellarx_explore_') as tmpdir:
    tmpdir = Path(tmpdir)

    test_config = {
        'dataset': {
            **ds_cfg,
            'image_width': 128,
            'image_height': 128,
            'num_train': 3,
            'num_val': 1,
            'num_test': 1,
            'output_dir': str(tmpdir / 'raw'),
            'metadata_file': str(tmpdir / 'raw' / 'metadata.json'),
        }
    }

    summary = build_dataset(test_config, verbose=True)

    # Reload and validate one image
    entries = load_metadata(test_config['dataset']['metadata_file'])
    first_entry = entries[0]
    img_path = tmpdir / 'raw' / first_entry['image_file']
    img = load_image(img_path)

    print(f'\nPipeline validation:')
    print(f'  Total entries in metadata : {len(entries)}')
    print(f'  First image path          : {first_entry["image_file"]}')
    print(f'  Image shape               : {img.shape}')
    print(f'  Image dtype               : {img.dtype}')
    print(f'  Image value range         : [{img.min():.4f}, {img.max():.4f}]')
    print(f'  Stars in first image      : {first_entry["n_stars"]}')
    print(f'  Boresight                 : RA={first_entry["boresight_ra_deg"]:.2f}°, '
          f'Dec={first_entry["boresight_dec_deg"]:+.2f}°')
    print('\n✅ Full pipeline test passed.')

---
## 8. Dataset statistics across a larger batch

In [ ]:
# Generate 20 images in memory (no disk write) for statistical analysis
batch_size = 20
batch_gen = StarFieldGenerator(catalog, ds_cfg)
batch = [batch_gen.generate(seed=i) for i in range(batch_size)]

batch_stats = {
    'n_stars_mean':  np.mean([len(sf.stars) for sf in batch]),
    'n_stars_std':   np.std([len(sf.stars)  for sf in batch]),
    'n_stars_min':   min(len(sf.stars) for sf in batch),
    'n_stars_max':   max(len(sf.stars) for sf in batch),
    'img_max_mean':  np.mean([sf.image.max() for sf in batch]),
    'img_mean_mean': np.mean([sf.image.mean() for sf in batch]),
}

print(f'Statistics over {batch_size} generated images (128×128, FoV={ds_cfg["field_of_view_deg"]}°):')
for k, v in batch_stats.items():
    print(f'  {k:<20s}: {v:.4f}')

---
## 9. Summary and Phase 2 preparation notes

**What Phase 1 has established:**

| Item | Status |
|---|---|
| Hipparcos bright-star catalog loaded (50 stars) | ✅ |
| Synthetic star-field generator implemented | ✅ |
| Reproducible (seed-controlled) generation | ✅ |
| Ground-truth metadata schema defined and validated | ✅ |
| 16-bit PNG save/load round-trip verified | ✅ |
| Full dataset builder (800 train / 100 val / 100 test) ready | ✅ |

**Observations for Phase 2 (Star Detection):**

- Stars appear as small Gaussian blobs (sigma ≈ 1.5 px) above a low background (~0.02).
- The brightest stars reach peak flux ≈ 0.9–1.0; the faintest rendered stars have flux ≈ 0.05.
- Background + read noise is approximately Gaussian; the image dynamic range is dominated by bright stars.
- The number of stars per frame varies significantly with boresight direction; some pointings
  yield 0 visible stars (the 50-star prototype catalog is sparse).
  **Recommendation for Phase 2:** extend the catalog to the full Hipparcos dataset (~118,000 stars)
  so that every pointing has a realistic star density.
- Detection thresholding should be calibrated against the background level recorded in config.yaml.

**Next notebook:** `02_star_detection.ipynb` — implement and tune the star detection algorithm.